In [1]:
import os
import json
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda

# ==========================================
# 1. Automatic Environment Detection
# ==========================================
try:
    # --- Colab Environment ---
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
    print("☁️ Colab Environment Detected.")
except ImportError:
    # --- Local Environment (VS Code) ---
    from dotenv import load_dotenv
    current_dir = os.getcwd()
    project_root = os.path.abspath(os.path.join(current_dir, "..", ".."))
    env_path = os.path.join(project_root, ".env")
    load_dotenv(dotenv_path=env_path, override=True)
    print("💻 Local Environment Detected.")

# ==========================================
# 2. Initialize the Model
# ==========================================
model = ChatGoogleGenerativeAI(
    model="gemini-flash-lite-latest",
    temperature=0.1
)
print("🚀 Gemini model is ready!")

💻 Local Environment Detected.
🚀 Gemini model is ready!


In [2]:
# ==========================================
# Defining Custom Python Functions
# ==========================================

def clean_text(text):
    """Removes extra spaces from the beginning and end of the text."""
    return text.strip()

def count_words(text):
    """Counts the number of words in the text and formats the output."""
    word_count = len(text.split())
    return f"متن: {text}\n\nتعداد کلمات: {word_count}"

print("✅ Custom Python functions are defined.")

✅ Custom Python functions are defined.


In [13]:
# ==========================================
# Injecting Python Logic into the Chain
# ==========================================

# Create the chain using RunnableLambda to wrap custom functions
chain = (
    ChatPromptTemplate.from_template("یک جمله کوتاه درباره شهر {city} بگو.")
    | model
    | StrOutputParser()
    | RunnableLambda(clean_text)   # Injection Step 1: Cleaning
    | RunnableLambda(count_words)  # Injection Step 2: Word Counting
)

print("⏳ Executing the chain with custom logic...\n")
result = chain.invoke({"city": "تهران"})
print(result)

⏳ Executing the chain with custom logic...

متن: تهران، کلان‌شهری پویا و پر از تضاد، که در سایه کوه توچال تاریخ و مدرنیته را به هم پیوند زده است.

تعداد کلمات: 21


In [19]:
# ==========================================
# Practical Use Case: Safely Extracting JSON
# ==========================================

def extract_json(text):
    """Safely extracts and parses a JSON block from the model's raw text output."""
    try:
        # Find the starting and ending brackets of the JSON block
        start = text.find("{")
        end = text.rfind("}") + 1
        json_str = text[start:end]
        
        # Convert JSON string to a Python Dictionary
        return json.loads(json_str) 
    except Exception as e:
        return {"error": "Could not parse JSON", "raw_text": text}

# Build the JSON extraction chain
json_chain = (
    ChatPromptTemplate.from_template("اطلاعات شهر {city} (نام کشور، جمعیت تقریبی) را فقط به صورت JSON برگردان.")
    | model
    | StrOutputParser()
    | RunnableLambda(extract_json) # Attach our custom JSON fixer
)

print("⏳ Executing JSON extraction chain...\n")
data = json_chain.invoke({"city": "تهران"})

# Display results and verify data type
print(f"Data Type: {type(data)}")

# Using .get() safely in case the model used English or Persian keys
country_name = data.get('کشور', data.get('country', 'نامشخص'))
print(f"Country: {country_name}")

print("\nFull Dictionary:", data)

⏳ Executing JSON extraction chain...

Data Type: <class 'dict'>
Country: ایران

Full Dictionary: {'city': 'تهران', 'country': 'ایران', 'population': 'تقریباً ۹ میلیون نفر (و بیش از ۱۴ میلیون نفر در کلان\u200cشهر تهران)'}
